In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri
from types import SimpleNamespace
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
from torch import nn
from torch.autograd import grad


In [ ]:
# Dataset helpers
def parabolic_inflow(y, U_max):
    y_t = torch.as_tensor(y, dtype=torch.float32)
    u = 4 * U_max * y_t * (0.41 - y_t) / (0.41**2)
    v = torch.zeros_like(y_t)
    return u, v

def get_dataset():
    data = np.load('data/ns_unsteady.npy', allow_pickle=True).item()
    u_ref = torch.from_numpy(data['u']).float()
    v_ref = torch.from_numpy(data['v']).float()
    p_ref = torch.from_numpy(data['p']).float()
    coords = torch.from_numpy(data['coords']).float()
    inflow_coords = torch.from_numpy(data['inflow_coords']).float()
    outflow_coords = torch.from_numpy(data['outflow_coords']).float()
    wall_coords = torch.from_numpy(data['wall_coords']).float()
    cylinder_coords = torch.from_numpy(data['cylinder_coords']).float()
    nu = torch.as_tensor(data['nu'], dtype=torch.float32)
    return (u_ref, v_ref, p_ref, coords, inflow_coords,
            outflow_coords, wall_coords, cylinder_coords, nu)


In [ ]:
# Model definition
class NavierStokes2D(nn.Module):
    def __init__(self, config, inflow_fn, temporal_dom, coords, Re):
        super().__init__()
        self.inflow_fn = inflow_fn
        self.temporal_dom = temporal_dom
        self.coords = coords
        self.Re = Re
        self.L = coords[:,0].max() - coords[:,0].min()
        self.W = coords[:,1].max() - coords[:,1].min()
        layers = []
        in_dim = 3
        hidden = config.arch.hidden_dim
        for _ in range(config.arch.num_layers - 1):
            layers.append(nn.Linear(in_dim, hidden))
            layers.append(nn.Tanh())
            in_dim = hidden
        layers.append(nn.Linear(in_dim, 3))
        self.network = nn.Sequential(*layers)

    def neural_net(self, t, x, y):
        t_n = t / self.temporal_dom[1]
        x_n = x / self.L
        y_n = y / self.W
        inputs = torch.stack([t_n, x_n, y_n], dim=-1)
        outputs = self.network(inputs)
        y_hat = y_n
        u = outputs[...,0] + 4 * 1.5 * y_hat * (0.41 - y_hat) / (0.41**2)
        v = outputs[...,1]
        p = outputs[...,2]
        return u, v, p

    def forward(self, t, x, y):
        return self.neural_net(t, x, y)

    def w_net(self, t, x, y):
        t.requires_grad_(True)
        x.requires_grad_(True)
        y.requires_grad_(True)
        u, v, _ = self.neural_net(t, x, y)
        u_y = grad(u, y, torch.ones_like(u), create_graph=True)[0]
        v_x = grad(v, x, torch.ones_like(v), create_graph=True)[0]
        return v_x - u_y

    def r_net(self, t, x, y):
        t.requires_grad_(True)
        x.requires_grad_(True)
        y.requires_grad_(True)
        u, v, p = self.neural_net(t, x, y)
        u_t = grad(u, t, torch.ones_like(u), create_graph=True)[0]
        v_t = grad(v, t, torch.ones_like(v), create_graph=True)[0]
        u_x = grad(u, x, torch.ones_like(u), create_graph=True)[0]
        v_x = grad(v, x, torch.ones_like(v), create_graph=True)[0]
        p_x = grad(p, x, torch.ones_like(p), create_graph=True)[0]
        u_y = grad(u, y, torch.ones_like(u), create_graph=True)[0]
        v_y = grad(v, y, torch.ones_like(v), create_graph=True)[0]
        p_y = grad(p, y, torch.ones_like(p), create_graph=True)[0]
        u_xx = grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
        v_xx = grad(v_x, x, torch.ones_like(v_x), create_graph=True)[0]
        u_yy = grad(u_y, y, torch.ones_like(u_y), create_graph=True)[0]
        v_yy = grad(v_y, y, torch.ones_like(v_y), create_graph=True)[0]
        ru = u_t + u*u_x + v*u_y + p_x - (u_xx + u_yy)/self.Re
        rv = v_t + u*v_x + v*v_y + p_y - (v_xx + v_yy)/self.Re
        rc = u_x + v_y
        u_out = u_x / self.Re - p
        v_out = v_x
        return ru, rv, rc, u_out, v_out


In [ ]:
# Training and evaluation helpers
def train_and_evaluate(config, workdir):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    (u_ref, v_ref, p_ref, coords, inflow_coords, outflow_coords, wall_coords, cylinder_coords, nu) = get_dataset()
    coords = coords.to(device)
    Re = 1.0 / nu.item() if nu.numel() == 1 else 1.0
    temporal_dom = torch.tensor([0.0, 1.0], device=device)
    inflow_fn = lambda y: parabolic_inflow(y, U_max=1.5)
    model = NavierStokes2D(config, inflow_fn, temporal_dom, coords, Re).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.optim.learning_rate)
    writer = SummaryWriter(Path(workdir) / config.wandb.name)
    for step in range(config.training.max_steps):
        t = torch.rand(config.training.res_batch_size, device=device) * temporal_dom[1]
        idx = torch.randint(0, coords.shape[0], (config.training.res_batch_size,), device=device)
        x = coords[idx,0]
        y = coords[idx,1]
        ru, rv, rc, _, _ = model.r_net(t, x, y)
        loss = ru.pow(2).mean() + rv.pow(2).mean() + rc.pow(2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % config.logging.log_every_steps == 0:
            writer.add_scalar('loss', loss.item(), step)
            print(f'step {step}: loss {loss.item():.6f}')
    writer.close()
    save_dir = Path(workdir) / 'ckpt' / config.wandb.name
    save_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), save_dir / 'model.pt')
    return model, temporal_dom, coords

def visualize_prediction(model, temporal_dom, coords):
    device = coords.device
    (u_ref, v_ref, p_ref, *_) = get_dataset()
    u_ref_end = u_ref[-1].to(device)
    v_ref_end = v_ref[-1].to(device)
    p_ref_end = p_ref[-1].to(device)
    with torch.no_grad():
        t = torch.tensor([temporal_dom[1]], device=device)
        u, v, p = model(t.repeat(coords.shape[0]), coords[:,0], coords[:,1])
    rel_u = torch.linalg.norm(u - u_ref_end) / torch.linalg.norm(u_ref_end)
    rel_v = torch.linalg.norm(v - v_ref_end) / torch.linalg.norm(v_ref_end)
    rel_p = torch.linalg.norm(p - p_ref_end) / torch.linalg.norm(p_ref_end)
    print(f'Relative L2 error u: {rel_u.item():.4f}, v: {rel_v.item():.4f}, p: {rel_p.item():.4f}')
    x = coords[:,0].cpu().numpy()
    y = coords[:,1].cpu().numpy()
    triang = tri.Triangulation(x, y)
    fig, axes = plt.subplots(3, 1, figsize=(6, 10))
    axes[0].tricontourf(triang, u.cpu().numpy(), levels=100, cmap='jet')
    axes[0].set_title('Predicted u')
    axes[1].tricontourf(triang, v.cpu().numpy(), levels=100, cmap='jet')
    axes[1].set_title('Predicted v')
    axes[2].tricontourf(triang, p.cpu().numpy(), levels=100, cmap='jet')
    axes[2].set_title('Predicted p')
    plt.tight_layout()
    plt.show()


In [ ]:
# Configure and run a short training loop
config = SimpleNamespace(
    wandb=SimpleNamespace(name='demo'),
    optim=SimpleNamespace(learning_rate=1e-3),
    training=SimpleNamespace(max_steps=1, res_batch_size=1024),
    logging=SimpleNamespace(log_every_steps=1),
    arch=SimpleNamespace(num_layers=4, hidden_dim=64),
)
workdir = '/tmp/ns_demo'
model, temporal_dom, coords = train_and_evaluate(config, workdir)
visualize_prediction(model, temporal_dom, coords)
